<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## 02-2 · Part 1: Vectors and Displacements

**A cooling decision is a vector; a displacement changes its early and late coordinates.**

Lecture 02-1 formulated the classroom cooling problem. This unit retains its state transition \\(F\\), performance mapping \\(G\\), and score mapping \\(H\\), with outdoor temperature fixed at 31 °C. Vector operations express changes in the same physical decision.

### 1 · Classroom system and optimization formulation

The real decision is how much cooling to use early and late. More cooling can reduce heat discomfort, but it uses energy and may make the room too cold.

Let \\(u=(u_{\mathrm{early}},u_{\mathrm{late}})\\). The horizon has \\(n=12\\) decision steps, with actions \\(u_0,\ldots,u_{11}\\) and states \\(T_0,\ldots,T_{12}\\).

> $\displaystyle u_t=\begin{cases}u_{\mathrm{early}},&t=0,\ldots,5,\\u_{\mathrm{late}},&t=6,\ldots,11.\end{cases}$

Indoor temperature \\(T_t\\) is produced by the system; it is not chosen directly. With initial temperature \\(T_0=27\,^{\circ}\mathrm C\\), the physical transition is

> $\displaystyle T_{t+1}=F(T_t,T_t^{\mathrm{out}},N_t,u_t;a,b,c)$
>
> $\displaystyle \phantom{T_{t+1}}=T_t+a(T_t^{\mathrm{out}}-T_t)+bN_t-cu_t,\quad t=0,\ldots,11.$

| Role | Values held fixed in the demonstrations |
|:---|:---|
| Fixed parameters | $(a,b,c)=(0.12,0.012,0.45)$ |
| External inputs | $T_t^{\mathrm{out}}=31\,^{\circ}\mathrm C$ and $N_t=20$ people at every step |
| Cooling limits | $u_{\min}=0$, $u_{\max}=5$ cooling units |
| State limits | $T_{\min}=20\,^{\circ}\mathrm C$, $T_{\max}=30\,^{\circ}\mathrm C$ |
| Energy limit | $E_{\max}=60$ model energy units |
| Energy-weight hyperparameter | $\lambda_E=1$ unless stated otherwise |

The performance mapping \\(G\\) measures discomfort outside the 22–24 °C comfort range and energy use:

> $\displaystyle D(u)=\sum_{t=1}^{12}\left[\max(T_t-24,0)^2+\max(22-T_t,0)^2\right].$
>
> $\displaystyle E(u)=\frac12\sum_{t=0}^{11}u_t^2=3\left(u_{\mathrm{early}}^2+u_{\mathrm{late}}^2\right).$

\\(D\\) uses squared-temperature step units. \\(E\\) uses model energy units, not calibrated kWh. The weight converts energy into the chosen score scale:

> $\displaystyle J(u;\lambda_E)=H(D(u),E(u);\lambda_E)=D(u)+\lambda_EE(u).$

Feasibility requires cooling bounds, all state limits for \\(t=1,\ldots,12\\), and \\(E(u)\le E_{\max}\\). The comfort range is a performance target; the wider 20–30 °C range is a hard requirement.

In standard notation, \\(x=[u_{\mathrm{early}},u_{\mathrm{late}}]^{\mathsf T}\\) and \\(y=\operatorname{Sim}(x)\\). Here \\(\operatorname{Sim}\\) composes repeated \\(F\\) transitions with \\(G\\). The objective \\(f(y;\lambda_E)\\) is the standard-form name for the score supplied by \\(H\\). Differentiation of \\(J\\) with respect to \\(u\\) includes its effect through the simulated states.


In [ ]:
import sys
import warnings

import matplotlib
import numpy as np
from matplotlib.patches import Patch, Rectangle
from matplotlib.lines import Line2D


def _pyplot(*, interactive=False):
    """Use ipympl outside the Playground, with a static fallback."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (ImportError, RuntimeError, ValueError):
            try:
                matplotlib.use("module://ipympl.backend_nbagg", force=True)
            except (ImportError, RuntimeError, ValueError):
                warnings.warn("Interactive backend unavailable; showing a static preview.")
    import matplotlib.pyplot as plt
    return plt


# Horizon and initial state
TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External inputs
OUTSIDE_TEMPERATURE = np.full(TIME_STEPS, 31.0)
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Requirement limits
MIN_COOLING, MAX_COOLING = 0.0, 5.0
MIN_TEMPERATURE, MAX_TEMPERATURE = 20.0, 30.0
MAX_ENERGY = 60.0

# Evaluation hyperparameter
ENERGY_WEIGHT = 1.0

# Stable visual roles
BLUE, TEAL, ORANGE = "#2563EB", "#0F8B7C", "#E88726"
PURPLE, GRAY = "#7C3AED", "#9CA3AF"


def style_axis(axis):
    axis.grid(alpha=0.25)
    axis.spines[["top", "right"]].set_visible(False)


def decision_axes(axis):
    axis.set(xlabel="Early cooling (cooling units)",
             ylabel="Late cooling (cooling units)", xlim=(0, 5), ylim=(0, 5))
    axis.set_aspect("equal")
    style_axis(axis)

Simulation produces the state path. The performance mapping calculates discomfort and energy; the requirement checks determine feasibility. Evaluation collects these quantities with the score in one result.


In [ ]:
def expand_decision(decision):
    """Expand the chosen levels into u_0, ..., u_11."""
    decision = np.asarray(decision, dtype=float)
    if decision.shape != (2,) or not np.isfinite(decision).all():
        raise ValueError("A decision must contain two finite cooling levels.")
    return np.repeat(decision, TIME_STEPS // 2)


def simulate_classroom(decision):
    cooling_schedule = expand_decision(decision)
    temperatures = np.empty(TIME_STEPS + 1)
    temperatures[0] = INITIAL_TEMPERATURE
    for t in range(TIME_STEPS):
        temperatures[t + 1] = (
            temperatures[t]
            + WEATHER_EXCHANGE * (OUTSIDE_TEMPERATURE[t] - temperatures[t])
            + OCCUPANT_HEAT * OCCUPANTS[t]
            - COOLING_EFFECT * cooling_schedule[t]
        )
    return cooling_schedule, temperatures


def performance_outputs(cooling_schedule, temperatures):
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling_schedule ** 2)
    return float(discomfort), float(energy)


def check_feasibility(decision, temperatures, energy):
    violations = []
    if not np.all((MIN_COOLING <= decision) & (decision <= MAX_COOLING)):
        violations.append("cooling bound")
    if np.min(temperatures[1:]) < MIN_TEMPERATURE:
        violations.append("minimum temperature")
    if np.max(temperatures[1:]) > MAX_TEMPERATURE:
        violations.append("maximum temperature")
    if energy > MAX_ENERGY:
        violations.append("energy limit")
    return tuple(violations)


def evaluate_candidate(decision, energy_weight=ENERGY_WEIGHT):
    decision = np.asarray(decision, dtype=float)
    cooling_schedule, temperatures = simulate_classroom(decision)
    discomfort, energy = performance_outputs(cooling_schedule, temperatures)
    violations = check_feasibility(decision, temperatures, energy)
    return {
        "decision": decision.copy(), "cooling_schedule": cooling_schedule,
        "temperatures": temperatures, "discomfort": discomfort, "energy": energy,
        "feasible": not violations, "violations": violations,
        "energy_weight": float(energy_weight),
        "objective": discomfort + energy_weight * energy,
    }


def score(decision, energy_weight=ENERGY_WEIGHT):
    """Also defined outside the feasible set for geometry and derivatives."""
    return evaluate_candidate(decision, energy_weight)["objective"]

Increasing early cooling from 3 to 3.2, with late cooling fixed at 2, lowers the final temperature. The comparison reports feasibility, raw performance outputs, and score for both decisions.


In [ ]:
for decision in [(3.0, 2.0), (3.2, 2.0)]:
    result = evaluate_candidate(decision)
    status = "feasible" if result["feasible"] else f"rejected: {result['violations']}"
    print(f"u={decision}: {status}; T_12={result['temperatures'][-1]:.2f} °C; "
          f"D={result['discomfort']:.2f}; E={result['energy']:.2f}; "
          f"J={result['objective']:.2f}")

### 2 · Decision vectors and displacements

A scalar is one number. A vector is an ordered collection of numbers. Here the order always means early cooling first, late cooling second. The two-coordinate space is \\(\mathbb R^2\\).

> $\displaystyle x=\begin{bmatrix}u_{\mathrm{early}}\\u_{\mathrm{late}}\end{bmatrix},\qquad x_0=\begin{bmatrix}3\\2\end{bmatrix}.$

The superscript \\(\mathsf T\\) means transpose: \\([3,2]^{\mathsf T}\\) writes the same column vector compactly. This column has shape \\(2\times1\\). In NumPy we use an array of shape <code>(2,)</code>; <code>decision[:, None]</code> makes an explicit column. Transposing a one-dimensional array with <code>.T</code> does not turn it into a column.

A point gives a location. A displacement vector \\(d\\) gives a change between locations. Addition and scalar multiplication act coordinate by coordinate:

> $\displaystyle u_{\mathrm{new}}=u+\alpha d=(u_{\mathrm{early}}+\alpha d_1,\ u_{\mathrm{late}}+\alpha d_2).$

Here \\(d=(0.5,-0.5)\\) increases early cooling and reduces late cooling. The scalar \\(\alpha\\) scales this change. It is a search hyperparameter; the resulting \\(u_{\mathrm{new}}\\) is the real-system decision. Coordinate directions \\(e_1=(1,0)\\) and \\(e_2=(0,1)\\) let us write \\(d=0.5e_1-0.5e_2\\).

At \\(\alpha=0.5\\), the proposed decision is \\(u_{\mathrm{new}}=(3.25,1.75)\\). Membership in \\([0,5]^2\\) establishes only the cooling bounds; temperature and energy limits impose additional requirements.

The decision-plane displacement corresponds to changes in the 12-step cooling schedule. The outlined square shows only the cooling bounds.

In [ ]:
def show_decision_vectors(decision=(3.0, 2.0), change=(0.5, -0.5), scale=1.0):
    plt = _pyplot()
    decision, change = np.asarray(decision), np.asarray(change)
    proposed = decision + scale * change
    figure, axes = plt.subplots(1, 2, figsize=(10.8, 4.6))
    axis = axes[0]
    axis.add_patch(Rectangle((0, 0), 5, 5, fill=False, edgecolor=BLUE, linewidth=1.5))
    axis.annotate("", xy=decision, xytext=(0, 0),
                  arrowprops=dict(arrowstyle="->", color=BLUE, lw=2.5))
    axis.annotate("", xy=proposed, xytext=decision,
                  arrowprops=dict(arrowstyle="->", color=ORANGE, lw=2.5))
    axis.plot([decision[0], decision[0], 0], [0, decision[1], decision[1]],
              "--", color=BLUE, alpha=0.5)
    axis.scatter(*decision, color=BLUE, s=65, label="Starting decision")
    axis.scatter(*proposed, color=ORANGE, s=80, label="Proposed decision", zorder=4)
    axis.text(0.35, 4.5, "Square = cooling bounds only", fontsize=10)
    decision_axes(axis)
    axis.set(xlim=(-0.1, 5.2), ylim=(-0.1, 5.2), title="A displacement changes two coordinates")
    axis.legend(loc="lower right", fontsize=9)
    for vector, color, label in [(decision, BLUE, "Starting decision"),
                                 (proposed, ORANGE, "Proposed decision")]:
        axes[1].stairs(expand_decision(vector), np.arange(13), color=color,
                       linewidth=2.5, label=label, baseline=None)
    axes[1].axvline(6, color=GRAY, linestyle=":")
    axes[1].set(xlabel="Decision-step boundary", ylabel="Cooling (cooling units)",
                xlim=(0, 12), ylim=(0, 5), xticks=np.arange(0, 13, 2),
                title="The vector expands into 12 actions")
    axes[1].legend(fontsize=9)
    style_axis(axes[1])
    figure.tight_layout()
    return figure

The orange displacement changes the early and late schedule levels. Each interval \\([t,t+1)\\) represents action \\(u_t\\).


<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-2_mathematics_for_optimization/assets/01_decision_vector_geometry.svg" alt="A displacement in early and late cooling coordinates and the corresponding twelve-step schedules" width="900" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

In [ ]:
decision = np.array([3.0, 2.0])
change = np.array([0.5, -0.5])
print("At scale 0.5:", decision + 0.5 * change)
print("Array shape:", decision.shape, "Column shape:", decision[:, None].shape)
vector_figure = show_decision_vectors()
_pyplot().show()
_pyplot().close(vector_figure)